# SAE Feature Decomposition

Use pre-trained Sparse Autoencoders to decompose a model's activations into interpretable features.

No SAELens dependency needed — mechkit loads the weights directly from HuggingFace.

In [ ]:
import mechkit

model = mechkit.load("gpt2")

## Using a synthetic SAE

For demonstration without downloading a real SAE, we can create a synthetic one.
In practice, you'd pass a HuggingFace repo ID like `"jbloom/GPT2-Small-SAEs-Reformatted"`.

In [ ]:
import torch
from mechkit.ops.sae import load_sae_from_tensors

# GPT-2's hidden size is 768
d_in = 768
d_sae = 4096  # typical SAE has many more features than the original dimension

sae = load_sae_from_tensors(
    W_enc=torch.randn(d_in, d_sae) * 0.01,
    W_dec=torch.randn(d_sae, d_in) * 0.01,
    b_enc=torch.zeros(d_sae),
    b_dec=torch.zeros(d_in),
)
print(f"SAE: {sae.d_in} -> {sae.d_sae} features")

## Decompose activations

In [ ]:
result = model.features(
    "The capital of France is",
    at="transformer.h.8",
    sae=sae,
    top_k=15,
)

## Inspect the result dict

In [ ]:
print(f"Reconstruction error: {result['reconstruction_error']:.4f}")
print(f"Sparsity: {result['sparsity']:.2%}")
print(f"Active features: {result['num_active_features']} / {result['total_features']}")
print(f"\nTop 5 features:")
for idx, val in result['top_features'][:5]:
    print(f"  Feature {idx}: activation = {val:.4f}")

## Using a real SAE from HuggingFace

To use a real pre-trained SAE, pass the HuggingFace repo ID directly.
The weights will be downloaded automatically.

```python
# This downloads ~100MB of SAE weights
result = model.features(
    "The capital of France is",
    at="transformer.h.8",
    sae="jbloom/GPT2-Small-SAEs-Reformatted",
)
```

## Compare features across prompts

See which features activate differently for related but distinct inputs.

In [ ]:
prompts = [
    "The capital of France is",
    "The capital of Germany is",
    "The largest city in Japan is",
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    r = model.features(prompt, at="transformer.h.8", sae=sae, top_k=5)